# Exp26 — Word order, calibrated

**What's new vs Exp25**
1. **Calibrated metric with a differential transfer index (DTI).** Each cell is measured against two foreign controls:
   - **Cross-opposite donor** (what Exp22/25 measured): foreign language whose canonical order is the opposite of L's. E.g., for EN adj (adj-N), the cross-opp donors are FR, ES, AR (all N-adj).
   - **Cross-same-order donor** (new): foreign language whose canonical order *matches* L's. E.g., for EN adj, the cross-same donors are DE, HI, ZH (all adj-N).
   
   Any shift seen under cross-same is "foreign-state injection noise", not order-specific transfer. **DTI = frac_flip(cross-opp) − frac_flip(cross-same)**. Positive DTI with CI excluding zero = order info transferred, net of the output-language confound.
2. **Multiple donors per cell** averaged within scene before scene-level bootstrap.
3. **Same-language floor donor** (same L, different scene) kept as a sanity check (should be ≈0).
4. **Expanded items**: 20 adj / 15 clause / 15 adp scenes.
5. **Bootstrap 95% CIs** on every number.
6. **CKA dropped** (uninterpretable at n<20 in high-dim space with the Gram-matrix-rank bias).
7. **R_retr guarded** against the identical-state degeneracy and marked `n/a` for adj (decision token precedes the scene).

**HI caveat**: For `clause`, HI is the only SOV language, so there is no cross-same donor for HI. DTI is NaN for HI/clause by design; the raw frac_flip is still reported.

**Interpretation**
- Positive DTI, CI excludes 0 → order transfers cross-lingually at this cell.
- DTI near 0 → whatever the cross-opp donor did, the cross-same donor did equally. Not evidence of order transfer.
- Floor should sit near 0. If it doesn't, the whole cell is suspect.

Fresh notebook. Results cache in `exp26_results/` so re-running the GPU cell skips completed cells. Untested on GPU in my sandbox; item structure and all analysis functions tested on synthetic data.

In [1]:
# CELL 1 — config
import os
GPU_ID = 2
os.environ.setdefault("CUDA_VISIBLE_DEVICES", str(GPU_ID))
for v in ("HTTP_PROXY","HTTPS_PROXY","ALL_PROXY","http_proxy","https_proxy","all_proxy"): os.environ.pop(v, None)
os.environ["HF_HUB_OFFLINE"] = "1"

import json, time, gc, itertools
from pathlib import Path
import numpy as np

MODELS = {
    "bloom-1b7": "bigscience/bloom-1b7",
    "bloom-7b1": "bigscience/bloom-7b1",
}
OUT = Path("exp26_results"); OUT.mkdir(exist_ok=True)
LAYER_STRIDE = 2
LANGS = ("en", "fr", "es", "de", "ar", "hi", "zh")

# Canonical order per (phenom, lang). "pre" = pre-order (adj-N, SVO, pre-position); "post" = reverse.
CANONICAL = {
    "adj":    {"en":"pre","de":"pre","zh":"pre","hi":"pre","fr":"post","es":"post","ar":"post"},
    "clause": {"en":"pre","fr":"pre","es":"pre","de":"pre","ar":"pre","zh":"pre","hi":"post"},
    "adp":    {"en":"pre","fr":"pre","es":"pre","de":"pre","ar":"pre","hi":"post","zh":"post"},
}
def opp_donors(phenom, L):
    want = "post" if CANONICAL[phenom][L] == "pre" else "pre"
    return tuple(l for l in LANGS if l != L and CANONICAL[phenom].get(l) == want)
def same_donors(phenom, L):
    want = CANONICAL[phenom][L]
    return tuple(l for l in LANGS if l != L and CANONICAL[phenom].get(l) == want)
print("config ok")

config ok


In [2]:
# CELL 2 — parallel items (20 adj / 15 clause / 15 adp)
# For every scene × language: (prefix, canonical_continuation, anti_continuation).
# Prefix must NOT end in a space. Continuations lead with space where tokenization needs it.
# c0 = canonical for that language, c1 = anti (reversed order).

PARALLEL_ADJ = [
  # ---- adj-N / N-adj: pre-order (EN DE HI ZH) vs post-order (FR ES AR) ----
  # 1 red house
  {"en":("The"," red house"," house red"),
   "fr":("La"," maison rouge"," rouge maison"),
   "es":("La"," casa roja"," roja casa"),
   "de":("Das"," rote Haus"," Haus rot"),
   "ar":("هذا","البيت الأحمر","الأحمر البيت"),  # VERIFY
   "hi":("यह"," लाल घर है"," घर लाल है"),         # VERIFY
   "zh":("这是","红房子","房子红")},               # VERIFY
  # 2 big dog
  {"en":("A"," big dog"," dog big"),
   "fr":("Un"," chien gros"," gros chien"),
   "es":("Un"," perro grande"," grande perro"),
   "de":("Ein"," großer Hund"," Hund groß"),
   "ar":("هذا","الكلب الكبير","الكبير الكلب"),   # VERIFY
   "hi":("यह"," बड़ा कुत्ता है"," कुत्ता बड़ा है"),  # VERIFY
   "zh":("这是","大狗","狗大")},                   # VERIFY
  # 3 new book
  {"en":("The"," new book"," book new"),
   "fr":("Le"," livre récent"," récent livre"),
   "es":("El"," libro nuevo"," nuevo libro"),
   "de":("Das"," neue Buch"," Buch neu"),
   "ar":("هذا","الكتاب الجديد","الجديد الكتاب"),  # VERIFY
   "hi":("यह"," नई किताब है"," किताब नई है"),      # VERIFY
   "zh":("这是","新书","书新")},                    # VERIFY
  # 4 small cat
  {"en":("A"," small cat"," cat small"),
   "fr":("Un"," chat petit"," petit chat"),
   "es":("Un"," gato pequeño"," pequeño gato"),
   "de":("Eine"," kleine Katze"," Katze klein"),
   "ar":("هذه","القطة الصغيرة","الصغيرة القطة"),   # VERIFY
   "hi":("यह"," छोटी बिल्ली है"," बिल्ली छोटी है"),  # VERIFY
   "zh":("这是","小猫","猫小")},                     # VERIFY
  # 5 black car
  {"en":("A"," black car"," car black"),
   "fr":("Une"," voiture noire"," noire voiture"),
   "es":("Un"," coche negro"," negro coche"),
   "de":("Ein"," schwarzes Auto"," Auto schwarz"),
   "ar":("هذه","السيارة السوداء","السوداء السيارة"),  # VERIFY
   "hi":("यह"," काली गाड़ी है"," गाड़ी काली है"),       # VERIFY
   "zh":("这是","黑车","车黑")},                        # VERIFY
  # 6 green tree
  {"en":("A"," green tree"," tree green"),
   "fr":("Un"," arbre vert"," vert arbre"),
   "es":("Un"," árbol verde"," verde árbol"),
   "de":("Ein"," grüner Baum"," Baum grün"),
   "ar":("هذه","الشجرة الخضراء","الخضراء الشجرة"),   # VERIFY
   "hi":("यह"," हरा पेड़ है"," पेड़ हरा है"),           # VERIFY
   "zh":("这是","绿树","树绿")},                        # VERIFY
  # 7 old man
  {"en":("An"," old man"," man old"),
   "fr":("Un"," homme âgé"," âgé homme"),
   "es":("Un"," hombre viejo"," viejo hombre"),
   "de":("Ein"," alter Mann"," Mann alt"),
   "ar":("هذا","الرجل العجوز","العجوز الرجل"),     # VERIFY
   "hi":("यह"," बूढ़ा आदमी है"," आदमी बूढ़ा है"),      # VERIFY
   "zh":("这是","老人","人老")},                      # VERIFY
  # 8 white bird
  {"en":("A"," white bird"," bird white"),
   "fr":("Un"," oiseau blanc"," blanc oiseau"),
   "es":("Un"," pájaro blanco"," blanco pájaro"),
   "de":("Ein"," weißer Vogel"," Vogel weiß"),
   "ar":("هذا","الطائر الأبيض","الأبيض الطائر"),   # VERIFY
   "hi":("यह"," सफेद पक्षी है"," पक्षी सफेद है"),      # VERIFY
   "zh":("这是","白鸟","鸟白")},                      # VERIFY
  # 9 young girl
  {"en":("A"," young girl"," girl young"),
   "fr":("Une"," fille mineure"," mineure fille"),
   "es":("Una"," chica joven"," joven chica"),
   "de":("Ein"," junges Mädchen"," Mädchen jung"),
   "ar":("هذه","الفتاة الصغيرة","الصغيرة الفتاة"), # VERIFY
   "hi":("यह"," जवान लड़की है"," लड़की जवान है"),     # VERIFY
   "zh":("这是","年轻女孩","女孩年轻")},               # VERIFY
  # 10 cold water
  {"en":("The"," cold water"," water cold"),
   "fr":("L'"," eau froide"," froide eau"),
   "es":("El"," agua fría"," fría agua"),
   "de":("Das"," kalte Wasser"," Wasser kalt"),
   "ar":("هذا","الماء البارد","البارد الماء"),      # VERIFY
   "hi":("यह"," ठंडा पानी है"," पानी ठंडा है"),       # VERIFY
   "zh":("这是","冷水","水冷")},                       # VERIFY
  # 11 tall tower
  {"en":("The"," tall tower"," tower tall"),
   "fr":("La"," tour haute"," haute tour"),
   "es":("La"," torre alta"," alta torre"),
   "de":("Der"," hohe Turm"," Turm hoch"),
   "ar":("هذا","البرج العالي","العالي البرج"),       # VERIFY
   "hi":("यह"," ऊंची मीनार है"," मीनार ऊंची है"),      # VERIFY
   "zh":("这是","高塔","塔高")},                        # VERIFY
  # 12 long road
  {"en":("A"," long road"," road long"),
   "fr":("Une"," route longue"," longue route"),
   "es":("Un"," camino largo"," largo camino"),
   "de":("Eine"," lange Straße"," Straße lang"),
   "ar":("هذا","الطريق الطويل","الطويل الطريق"),      # VERIFY
   "hi":("यह"," लंबी सड़क है"," सड़क लंबी है"),           # VERIFY
   "zh":("这是","长路","路长")},                          # VERIFY
  # 13 hot coffee
  {"en":("The"," hot coffee"," coffee hot"),
   "fr":("Le"," café chaud"," chaud café"),
   "es":("El"," café caliente"," caliente café"),
   "de":("Der"," heiße Kaffee"," Kaffee heiß"),
   "ar":("هذه","القهوة الساخنة","الساخنة القهوة"),    # VERIFY
   "hi":("यह"," गरम कॉफी है"," कॉफी गरम है"),           # VERIFY
   "zh":("这是","热咖啡","咖啡热")},                      # VERIFY
  # 14 heavy box
  {"en":("A"," heavy box"," box heavy"),
   "fr":("Une"," boîte lourde"," lourde boîte"),
   "es":("Una"," caja pesada"," pesada caja"),
   "de":("Eine"," schwere Kiste"," Kiste schwer"),
   "ar":("هذا","الصندوق الثقيل","الثقيل الصندوق"),    # VERIFY
   "hi":("यह"," भारी डिब्बा है"," डिब्बा भारी है"),       # VERIFY
   "zh":("这是","重箱","箱重")},                          # VERIFY
  # 15 good friend
  {"en":("A"," good friend"," friend good"),
   "fr":("Un"," ami fidèle"," fidèle ami"),
   "es":("Un"," amigo bueno"," bueno amigo"),
   "de":("Ein"," guter Freund"," Freund gut"),
   "ar":("هذا","الصديق الجيد","الجيد الصديق"),         # VERIFY
   "hi":("यह"," अच्छा दोस्त है"," दोस्त अच्छा है"),         # VERIFY
   "zh":("这是","好朋友","朋友好")},                        # VERIFY
  # 16 happy child
  {"en":("A"," happy child"," child happy"),
   "fr":("Un"," enfant content"," content enfant"),
   "es":("Un"," niño feliz"," feliz niño"),
   "de":("Ein"," glückliches Kind"," Kind glücklich"),
   "ar":("هذا","الطفل السعيد","السعيد الطفل"),         # VERIFY
   "hi":("यह"," खुश बच्चा है"," बच्चा खुश है"),           # VERIFY
   "zh":("这是","快乐孩子","孩子快乐")},                    # VERIFY
  # 17 beautiful river
  {"en":("A"," beautiful river"," river beautiful"),
   "fr":("Une"," rivière belle"," belle rivière"),
   "es":("Un"," río hermoso"," hermoso río"),
   "de":("Ein"," schöner Fluss"," Fluss schön"),
   "ar":("هذا","النهر الجميل","الجميل النهر"),         # VERIFY
   "hi":("यह"," सुंदर नदी है"," नदी सुंदर है"),            # VERIFY
   "zh":("这是","美丽河流","河流美丽")},                     # VERIFY
  # 18 sharp knife
  {"en":("A"," sharp knife"," knife sharp"),
   "fr":("Un"," couteau tranchant"," tranchant couteau"),
   "es":("Un"," cuchillo afilado"," afilado cuchillo"),
   "de":("Ein"," scharfes Messer"," Messer scharf"),
   "ar":("هذا","السكين الحاد","الحاد السكين"),         # VERIFY
   "hi":("यह"," तेज़ चाकू है"," चाकू तेज़ है"),           # VERIFY
   "zh":("这是","尖刀","刀尖")},                           # VERIFY
  # 19 strange sound
  {"en":("A"," strange sound"," sound strange"),
   "fr":("Un"," son étrange"," étrange son"),
   "es":("Un"," sonido raro"," raro sonido"),
   "de":("Ein"," seltsamer Klang"," Klang seltsam"),
   "ar":("هذا","الصوت الغريب","الغريب الصوت"),         # VERIFY
   "hi":("यह"," अजीब आवाज़ है"," आवाज़ अजीब है"),         # VERIFY
   "zh":("这是","奇怪声音","声音奇怪")},                     # VERIFY
  # 20 dark room
  {"en":("A"," dark room"," room dark"),
   "fr":("Une"," chambre sombre"," sombre chambre"),
   "es":("Un"," cuarto oscuro"," oscuro cuarto"),
   "de":("Ein"," dunkles Zimmer"," Zimmer dunkel"),
   "ar":("هذه","الغرفة المظلمة","المظلمة الغرفة"),     # VERIFY
   "hi":("यह"," अंधेरा कमरा है"," कमरा अंधेरा है"),        # VERIFY
   "zh":("这是","暗房间","房间暗")},                        # VERIFY
]

# ---- clause: SVO (EN FR ES DE AR ZH) vs SOV (HI). Simple main clauses only (no V2 trigger for DE). ----
PARALLEL_CLAUSE = [
  # 1
  {"en":("John"," sees the dog"," the dog sees"),
   "fr":("Jean"," voit le chien"," le chien voit"),
   "es":("Juan"," ve al perro"," al perro ve"),
   "de":("Peter"," sieht den Hund"," den Hund sieht"),
   "ar":("أحمد"," يرى الكلب"," الكلب يرى"),
   "hi":("जॉन"," कुत्ते को देखता है"," देखता है कुत्ते को"),
   "zh":("小明","看狗","狗看")},
  # 2
  {"en":("Mary"," reads a book"," a book reads"),
   "fr":("Marie"," lit un livre"," un livre lit"),
   "es":("María"," lee un libro"," un libro lee"),
   "de":("Anna"," liest ein Buch"," ein Buch liest"),
   "ar":("مريم"," تقرأ كتاباً"," كتاباً تقرأ"),
   "hi":("मेरी"," एक किताब पढ़ती है"," पढ़ती है एक किताब"),
   "zh":("小丽","读书","书读")},
  # 3
  {"en":("The boy"," eats the apple"," the apple eats"),
   "fr":("Le garçon"," mange la pomme"," la pomme mange"),
   "es":("El niño"," come la manzana"," la manzana come"),
   "de":("Der Junge"," isst den Apfel"," den Apfel isst"),
   "ar":("الولد"," يأكل التفاحة"," التفاحة يأكل"),
   "hi":("लड़का"," सेब खाता है"," खाता है सेब"),
   "zh":("男孩","吃苹果","苹果吃")},
  # 4
  {"en":("The woman"," buys bread"," bread buys"),
   "fr":("La femme"," achète du pain"," du pain achète"),
   "es":("La mujer"," compra pan"," pan compra"),
   "de":("Die Frau"," kauft Brot"," Brot kauft"),
   "ar":("المرأة"," تشتري الخبز"," الخبز تشتري"),
   "hi":("औरत"," रोटी खरीदती है"," खरीदती है रोटी"),
   "zh":("女人","买面包","面包买")},
  # 5
  {"en":("The student"," writes the essay"," the essay writes"),
   "fr":("L'étudiant"," écrit l'essai"," l'essai écrit"),
   "es":("El estudiante"," escribe el ensayo"," el ensayo escribe"),
   "de":("Der Student"," schreibt den Aufsatz"," den Aufsatz schreibt"),
   "ar":("الطالب"," يكتب المقال"," المقال يكتب"),
   "hi":("छात्र"," निबंध लिखता है"," लिखता है निबंध"),
   "zh":("学生","写文章","文章写")},
  # 6
  {"en":("The teacher"," helps the child"," the child helps"),
   "fr":("Le professeur"," aide l'enfant"," l'enfant aide"),
   "es":("El maestro"," ayuda al niño"," al niño ayuda"),
   "de":("Der Lehrer"," hilft dem Kind"," dem Kind hilft"),
   "ar":("المعلم"," يساعد الطفل"," الطفل يساعد"),
   "hi":("शिक्षक"," बच्चे की मदद करता है"," मदद करता है बच्चे की"),
   "zh":("老师","帮助孩子","孩子帮助")},
  # 7
  {"en":("The dog"," chases the cat"," the cat chases"),
   "fr":("Le chien"," poursuit le chat"," le chat poursuit"),
   "es":("El perro"," persigue al gato"," al gato persigue"),
   "de":("Der Hund"," jagt die Katze"," die Katze jagt"),
   "ar":("الكلب"," يطارد القطة"," القطة يطارد"),
   "hi":("कुत्ता"," बिल्ली का पीछा करता है"," पीछा करता है बिल्ली का"),
   "zh":("狗","追猫","猫追")},
  # 8
  {"en":("The man"," opens the door"," the door opens"),
   "fr":("L'homme"," ouvre la porte"," la porte ouvre"),
   "es":("El hombre"," abre la puerta"," la puerta abre"),
   "de":("Der Mann"," öffnet die Tür"," die Tür öffnet"),
   "ar":("الرجل"," يفتح الباب"," الباب يفتح"),
   "hi":("आदमी"," दरवाज़ा खोलता है"," खोलता है दरवाज़ा"),
   "zh":("男人","开门","门开")},
  # 9
  {"en":("The baker"," makes bread"," bread makes"),
   "fr":("Le boulanger"," fait du pain"," du pain fait"),
   "es":("El panadero"," hace pan"," pan hace"),
   "de":("Der Bäcker"," macht Brot"," Brot macht"),
   "ar":("الخباز"," يصنع الخبز"," الخبز يصنع"),
   "hi":("बेकर"," रोटी बनाता है"," बनाता है रोटी"),
   "zh":("面包师","做面包","面包做")},
  # 10
  {"en":("The farmer"," plants wheat"," wheat plants"),
   "fr":("Le fermier"," plante du blé"," du blé plante"),
   "es":("El granjero"," planta trigo"," trigo planta"),
   "de":("Der Bauer"," pflanzt Weizen"," Weizen pflanzt"),
   "ar":("المزارع"," يزرع القمح"," القمح يزرع"),
   "hi":("किसान"," गेहूं बोता है"," बोता है गेहूं"),
   "zh":("农夫","种小麦","小麦种")},
  # 11
  {"en":("The child"," draws a picture"," a picture draws"),
   "fr":("L'enfant"," dessine une image"," une image dessine"),
   "es":("El niño"," dibuja una imagen"," una imagen dibuja"),
   "de":("Das Kind"," zeichnet ein Bild"," ein Bild zeichnet"),
   "ar":("الطفل"," يرسم صورة"," صورة يرسم"),
   "hi":("बच्चा"," एक चित्र बनाता है"," बनाता है एक चित्र"),
   "zh":("小孩","画画","画画儿")},  # NB: cand1 for zh is weak; VERIFY
  # 12
  {"en":("The driver"," stops the car"," the car stops"),
   "fr":("Le conducteur"," arrête la voiture"," la voiture arrête"),
   "es":("El conductor"," para el coche"," el coche para"),
   "de":("Der Fahrer"," stoppt das Auto"," das Auto stoppt"),
   "ar":("السائق"," يوقف السيارة"," السيارة يوقف"),
   "hi":("ड्राइवर"," गाड़ी रोकता है"," रोकता है गाड़ी"),
   "zh":("司机","停车","车停")},
  # 13
  {"en":("The guest"," brings a gift"," a gift brings"),
   "fr":("L'invité"," apporte un cadeau"," un cadeau apporte"),
   "es":("El invitado"," trae un regalo"," un regalo trae"),
   "de":("Der Gast"," bringt ein Geschenk"," ein Geschenk bringt"),
   "ar":("الضيف"," يحضر هدية"," هدية يحضر"),
   "hi":("मेहमान"," एक तोहफा लाता है"," लाता है एक तोहफा"),
   "zh":("客人","带礼物","礼物带")},
  # 14
  {"en":("The nurse"," checks the patient"," the patient checks"),
   "fr":("L'infirmier"," examine le patient"," le patient examine"),
   "es":("La enfermera"," revisa al paciente"," al paciente revisa"),
   "de":("Die Krankenschwester"," prüft den Patienten"," den Patienten prüft"),
   "ar":("الممرضة"," تفحص المريض"," المريض تفحص"),
   "hi":("नर्स"," मरीज़ की जांच करती है"," जांच करती है मरीज़ की"),
   "zh":("护士","检查病人","病人检查")},
  # 15
  {"en":("The painter"," paints the wall"," the wall paints"),
   "fr":("Le peintre"," peint le mur"," le mur peint"),
   "es":("El pintor"," pinta la pared"," la pared pinta"),
   "de":("Der Maler"," streicht die Wand"," die Wand streicht"),
   "ar":("الرسام"," يطلي الجدار"," الجدار يطلي"),
   "hi":("चित्रकार"," दीवार रंगता है"," रंगता है दीवार"),
   "zh":("画家","刷墙","墙刷")},
]

# ---- adp: pre (EN FR ES DE AR) vs post (HI ZH-ish via location+里/上/下 pattern) ----
# Concrete locative/directional prepositions only.
PARALLEL_ADP = [
  # 1 in the house
  {"en":("He is",     " in the house",   " the house in"),
   "fr":("Il est",    " dans la maison", " la maison dans"),
   "es":("Está",      " en la casa",     " la casa en"),
   "de":("Er ist",    " im Haus",        " Haus im"),
   "ar":("هو",         " في البيت",       " البيت في"),
   "hi":("वह"," घर में है"," में घर है"),
   "zh":("他",         "在房子里",         "里房子在")},
  # 2 on the table
  {"en":("The book is"," on the table"," the table on"),
   "fr":("Le livre est"," sur la table"," la table sur"),
   "es":("El libro está"," sobre la mesa"," la mesa sobre"),
   "de":("Das Buch ist"," auf dem Tisch"," dem Tisch auf"),
   "ar":("الكتاب"," على الطاولة"," الطاولة على"),
   "hi":("किताब"," मेज़ पर है"," पर मेज़ है"),
   "zh":("书",         "在桌子上",         "上桌子在")},
  # 3 under the tree
  {"en":("The dog is"," under the tree"," the tree under"),
   "fr":("Le chien est"," sous l'arbre"," l'arbre sous"),
   "es":("El perro está"," bajo el árbol"," el árbol bajo"),
   "de":("Der Hund ist"," unter dem Baum"," dem Baum unter"),
   "ar":("الكلب"," تحت الشجرة"," الشجرة تحت"),
   "hi":("कुत्ता"," पेड़ के नीचे है"," नीचे पेड़ के है"),
   "zh":("狗",         "在树下",           "下树在")},
  # 4 near the river
  {"en":("The city is"," near the river"," the river near"),
   "fr":("La ville est"," près du fleuve"," du fleuve près"),
   "es":("La ciudad está"," cerca del río"," del río cerca"),
   "de":("Die Stadt liegt"," nahe dem Fluss"," dem Fluss nahe"),
   "ar":("المدينة"," قرب النهر"," النهر قرب"),
   "hi":("शहर"," नदी के पास है"," पास नदी के है"),
   "zh":("城市",       "在河边",           "边河在")},
  # 5 behind the wall
  {"en":("He stood"," behind the wall"," the wall behind"),
   "fr":("Il se tenait"," derrière le mur"," le mur derrière"),
   "es":("Estaba"," detrás del muro"," del muro detrás"),
   "de":("Er stand"," hinter der Mauer"," der Mauer hinter"),
   "ar":("وقف"," خلف الجدار"," الجدار خلف"),
   "hi":("वह"," दीवार के पीछे खड़ा था"," पीछे दीवार के खड़ा था"),
   "zh":("他",         "在墙后",           "后墙在")},
  # 6 inside the box
  {"en":("The key is"," inside the box"," the box inside"),
   "fr":("La clé est"," dans la boîte"," la boîte dans"),
   "es":("La llave está"," dentro de la caja"," de la caja dentro"),
   "de":("Der Schlüssel ist"," in der Kiste"," der Kiste in"),
   "ar":("المفتاح"," داخل الصندوق"," الصندوق داخل"),
   "hi":("चाबी"," डिब्बे के अंदर है"," अंदर डिब्बे के है"),
   "zh":("钥匙",       "在盒子里",         "里盒子在")},
  # 7 above the clouds
  {"en":("The plane is"," above the clouds"," the clouds above"),
   "fr":("L'avion est"," au-dessus des nuages"," des nuages au-dessus"),
   "es":("El avión está"," sobre las nubes"," las nubes sobre"),
   "de":("Das Flugzeug ist"," über den Wolken"," den Wolken über"),
   "ar":("الطائرة"," فوق السحاب"," السحاب فوق"),
   "hi":("विमान"," बादलों के ऊपर है"," ऊपर बादलों के है"),
   "zh":("飞机",       "在云上",           "上云在")},
  # 8 between the trees
  {"en":("The path runs"," between the trees"," the trees between"),
   "fr":("Le chemin passe"," entre les arbres"," les arbres entre"),
   "es":("El sendero va"," entre los árboles"," los árboles entre"),
   "de":("Der Weg führt"," zwischen den Bäumen"," den Bäumen zwischen"),
   "ar":("الطريق"," بين الأشجار"," الأشجار بين"),
   "hi":("रास्ता"," पेड़ों के बीच है"," बीच पेड़ों के है"),
   "zh":("小路",       "在树之间",         "之间树在")},
  # 9 beside the river
  {"en":("The hut sits"," beside the river"," the river beside"),
   "fr":("La hutte est"," au bord du fleuve"," du fleuve au bord"),
   "es":("La choza está"," junto al río"," al río junto"),
   "de":("Die Hütte steht"," neben dem Fluss"," dem Fluss neben"),
   "ar":("الكوخ"," بجانب النهر"," النهر بجانب"),
   "hi":("झोंपड़ी"," नदी के बगल में है"," बगल में नदी के है"),
   "zh":("小屋",       "在河旁",           "旁河在")},
  # 10 across the street
  {"en":("The shop is"," across the street"," the street across"),
   "fr":("Le magasin est"," en face de la rue"," de la rue en face"),
   "es":("La tienda está"," al otro lado de la calle"," de la calle al otro lado"),
   "de":("Der Laden ist"," gegenüber der Straße"," der Straße gegenüber"),
   "ar":("المتجر"," عبر الشارع"," الشارع عبر"),
   "hi":("दुकान"," सड़क के पार है"," पार सड़क के है"),
   "zh":("商店",       "在街对面",         "对面街在")},
  # 11 before the door
  {"en":("He stands"," before the door"," the door before"),
   "fr":("Il se tient"," devant la porte"," la porte devant"),
   "es":("Está"," ante la puerta"," la puerta ante"),
   "de":("Er steht"," vor der Tür"," der Tür vor"),
   "ar":("يقف"," أمام الباب"," الباب أمام"),
   "hi":("वह"," दरवाज़े के सामने खड़ा है"," सामने दरवाज़े के खड़ा है"),
   "zh":("他",         "在门前",           "前门在")},
  # 12 toward the city
  {"en":("He walks"," toward the city"," the city toward"),
   "fr":("Il marche"," vers la ville"," la ville vers"),
   "es":("Camina"," hacia la ciudad"," la ciudad hacia"),
   "de":("Er geht"," zur Stadt"," Stadt zur"),
   "ar":("يمشي"," نحو المدينة"," المدينة نحو"),
   "hi":("वह"," शहर की ओर जाता है"," ओर शहर की जाता है"),
   "zh":("他",         "朝城里走",         "城里朝走")},
  # 13 through the forest
  {"en":("The road goes"," through the forest"," the forest through"),
   "fr":("La route passe"," à travers la forêt"," la forêt à travers"),
   "es":("La carretera va"," por el bosque"," el bosque por"),
   "de":("Die Straße führt"," durch den Wald"," den Wald durch"),
   "ar":("الطريق"," عبر الغابة"," الغابة عبر"),
   "hi":("सड़क"," जंगल से होकर जाती है"," होकर जंगल से जाती है"),
   "zh":("路",         "穿过森林",         "森林穿过")},
  # 14 along the path
  {"en":("He walks"," along the path"," the path along"),
   "fr":("Il marche"," le long du chemin"," du chemin le long"),
   "es":("Camina"," a lo largo del sendero"," del sendero a lo largo"),
   "de":("Er geht"," entlang des Weges"," des Weges entlang"),
   "ar":("يمشي"," على طول المسار"," المسار على طول"),
   "hi":("वह"," रास्ते के साथ चलता है"," साथ रास्ते के चलता है"),
   "zh":("他",         "沿着路走",         "路沿着走")},
  # 15 against the wall
  {"en":("He leans"," against the wall"," the wall against"),
   "fr":("Il s'appuie"," contre le mur"," le mur contre"),
   "es":("Se apoya"," contra la pared"," la pared contra"),
   "de":("Er lehnt"," gegen die Wand"," die Wand gegen"),
   "ar":("يستند"," إلى الجدار"," الجدار إلى"),
   "hi":("वह"," दीवार के सहारे खड़ा है"," सहारे दीवार के खड़ा है"),
   "zh":("他",         "靠在墙上",         "墙上靠")},
]

PARALLEL = {"adj": PARALLEL_ADJ, "clause": PARALLEL_CLAUSE, "adp": PARALLEL_ADP}


def same_donors(phenom, L):
    want = CANONICAL[phenom][L]
    return tuple(l for l in LANGS if l != L and CANONICAL[phenom].get(l) == want)



def enc(tok, text): return tok(text, add_special_tokens=False)["input_ids"]
def new_tokens(tok, prefix, cont):
    a, b = enc(tok, prefix), enc(tok, prefix + cont)
    return b[len(a):] if b[:len(a)] == a else None
def item_valid(tok, scene, lang):
    if lang not in scene: return False
    pr, c0, c1 = scene[lang]
    if not pr or c0 == c1: return False
    t0, t1 = new_tokens(tok, pr, c0), new_tokens(tok, pr, c1)
    return bool(t0) and bool(t1) and t0 != t1

print({k: len(v) for k, v in PARALLEL.items()})

{'adj': 20, 'clause': 15, 'adp': 15}


In [3]:
# CELL 3 — analysis helpers (tested on synthetic data; see Exp26 handoff)
def transfer_stats(P, m_clean, boots=1000, seed=0):
    """P, m_clean: [n_scenes]. frac_flip > 0 means donor moved recipient from canonical toward anti."""
    P = np.asarray(P, float); m_clean = np.asarray(m_clean, float)
    diff = P - m_clean
    denom = 2 * np.abs(m_clean).mean() + 1e-9
    rng = np.random.default_rng(seed); bs = []
    for _ in range(boots):
        ix = rng.integers(0, len(diff), len(diff))
        bs.append(diff[ix].mean() / (2 * np.abs(m_clean[ix]).mean() + 1e-9))
    bs = np.array(bs)
    return dict(frac_flip=float(diff.mean() / denom),
                lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)))

def dti_stats(P_opp, P_same, m_clean, boots=1000, seed=0):
    """DTI = frac_flip(cross-opposite) - frac_flip(cross-same-order). Order-specific transfer net of foreign-state effects."""
    P_opp = np.asarray(P_opp, float); P_same = np.asarray(P_same, float); m_clean = np.asarray(m_clean, float)
    d_opp, d_same = P_opp - m_clean, P_same - m_clean
    denom = 2 * np.abs(m_clean).mean() + 1e-9
    rng = np.random.default_rng(seed); bs = []
    for _ in range(boots):
        ix = rng.integers(0, len(m_clean), len(m_clean))
        d = 2 * np.abs(m_clean[ix]).mean() + 1e-9
        bs.append((d_opp[ix].mean() - d_same[ix].mean()) / d)
    bs = np.array(bs)
    return dict(dti=float((d_opp.mean() - d_same.mean()) / denom),
                lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)))

def retrieval(HA, HL):
    rows = []; n, nl = HL.shape[0], HL.shape[1]
    for l in range(nl):
        A = HA[:, l] - HA[:, l].mean(0); B = HL[:, l] - HL[:, l].mean(0)
        deg = max(float((np.linalg.norm(A, axis=1) < 1e-3).mean()),
                  float((np.linalg.norm(B, axis=1) < 1e-3).mean()))
        if deg > 0.5:
            rows.append(dict(layer=l, top1=np.nan, R_retr=np.nan, mrr=np.nan)); continue
        A /= (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)
        B /= (np.linalg.norm(B, axis=1, keepdims=True) + 1e-9)
        S = B @ A.T; rank = (S > np.diag(S)[:, None]).sum(1) + 1
        top1 = float((rank == 1).mean())
        rows.append(dict(layer=l, top1=top1,
                         R_retr=(top1 - 1/n) / (1 - 1/n) if n > 1 else np.nan,
                         mrr=float((1/rank).mean())))
    return rows
print("analysis helpers ok")

analysis helpers ok


In [4]:
# CELL 4 — GPU run: capture states + patch floor / cross-same / cross-opp donors
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
dev = "cuda" if torch.cuda.is_available() else "cpu"

def get_modules(model):
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        bl = list(model.transformer.h); return dict(resid=bl, ln_f=model.transformer.ln_f, lm_head=model.lm_head)
    bl = list(model.model.layers); return dict(resid=bl, ln_f=model.model.norm, lm_head=model.lm_head)

def patch_hook(p, vec):
    def hook(mod, inp, out):
        tup = isinstance(out, tuple); o = (out[0] if tup else out).clone()
        v = vec if isinstance(vec, torch.Tensor) else torch.as_tensor(vec)
        o[:, p] = v.to(device=o.device, dtype=o.dtype)
        return (o,) + tuple(out[1:]) if tup else o
    return hook

@torch.no_grad()
def margin(model, mods, tok, bos, prefix, c0t, c1t, patch=None):
    ids = bos + enc(tok, prefix); p = len(ids) - 1
    rows = [ids + c0t, ids + c1t]; T = max(map(len, rows))
    pad = tok.pad_token_id if tok.pad_token_id is not None else 0
    inp = torch.tensor([r + [pad] * (T - len(r)) for r in rows], device=dev)
    att = torch.tensor([[1] * len(r) + [0] * (T - len(r)) for r in rows], device=dev)
    h = mods["resid"][patch[0]].register_forward_hook(patch_hook(p, patch[1])) if patch else None
    try: logits = model(input_ids=inp, attention_mask=att).logits
    finally:
        if h: h.remove()
    lp = []
    for r, ct in enumerate((c0t, c1t)):
        g = torch.log_softmax(logits[r, p:p + len(ct)].float(), -1)
        lp.append(sum(g[j, t].item() for j, t in enumerate(ct)))
    return lp[1] - lp[0]   # >0 means c1 (anti) preferred; <0 means c0 (canonical) preferred

@torch.no_grad()
def capture(model, mods, tok, bos, prefix):
    ids = bos + enc(tok, prefix); p = len(ids) - 1
    nl = len(mods["resid"]); store = [None] * nl; hs = []
    for li, m in enumerate(mods["resid"]):
        def hk(mod, inp, out, li=li):
            o = out[0] if isinstance(out, tuple) else out
            store[li] = o[0, p].detach()
        hs.append(m.register_forward_hook(hk))
    try: model(input_ids=torch.tensor([ids], device=dev))
    finally:
        for h in hs: h.remove()
    return torch.stack(store)

for key, path in MODELS.items():
    tok = AutoTokenizer.from_pretrained(path)
    bos = [tok.bos_token_id] if "bloom" not in key else []
    model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.bfloat16).to(dev).eval()
    mods = get_modules(model)
    nl = len(mods["resid"]); H = mods["lm_head"].weight.shape[1]
    LAYERS = sorted(set(list(range(0, nl, LAYER_STRIDE)) + [nl - 1]))

    for phenom in ("adj", "clause", "adp"):
        for L in LANGS:
            f = OUT / f"{key}__{phenom}__{L}.npz"
            if f.exists(): print("cached", f.name); continue
            scene_ids = [i for i, sc in enumerate(PARALLEL[phenom]) if item_valid(tok, sc, L)]
            if len(scene_ids) < 4: print("SKIP", key, phenom, L, "n=", len(scene_ids)); continue
            opp_D = [d for d in opp_donors(phenom, L) if any(item_valid(tok, PARALLEL[phenom][s], d) for s in scene_ids)]
            same_D = [d for d in same_donors(phenom, L) if any(item_valid(tok, PARALLEL[phenom][s], d) for s in scene_ids)]
            if not opp_D: print("SKIP", key, phenom, L, "no opp donor"); continue

            t0 = time.time(); n = len(scene_ids)
            m_clean = np.zeros(n)
            P_floor = np.full((n, len(LAYERS)), np.nan)
            P_opp  = np.full((n, max(1, len(opp_D)),  len(LAYERS)), np.nan)
            P_same = np.full((n, max(1, len(same_D)), len(LAYERS)), np.nan) if same_D else None
            HL = np.zeros((n, nl, H), dtype=np.float32)
            HA_opp = np.zeros_like(HL)  # anchor states for R_retr (first opp donor)
            cache = {}
            def cap_t(prefix):
                if prefix not in cache: cache[prefix] = capture(model, mods, tok, bos, prefix)
                return cache[prefix]
            def cap(prefix):
                return cap_t(prefix).float().cpu().numpy()

            # 1) baselines + capture states
            per_scene_meta = []
            for i, s in enumerate(scene_ids):
                sc = PARALLEL[phenom][s]
                pr_L, c0_L, c1_L = sc[L]
                c0t = new_tokens(tok, pr_L, c0_L); c1t = new_tokens(tok, pr_L, c1_L)
                m_clean[i] = margin(model, mods, tok, bos, pr_L, c0t, c1t)
                HL[i] = cap(pr_L)
                if opp_D: HA_opp[i] = cap(PARALLEL[phenom][s][opp_D[0]][0])
                per_scene_meta.append((s, pr_L, c0t, c1t))

            # 2) floor: same L, different scene
            for i, (s, pr_L, c0t, c1t) in enumerate(per_scene_meta):
                s_other = scene_ids[(i + 1) % n]
                pr_other = PARALLEL[phenom][s_other][L][0]
                donor = cap_t(pr_other)
                for j, l in enumerate(LAYERS):
                    P_floor[i, j] = margin(model, mods, tok, bos, pr_L, c0t, c1t, patch=(l, donor[l]))

            # 3) cross-opp donors (one state per donor, patched into recipient)
            for di, D in enumerate(opp_D):
                for i, (s, pr_L, c0t, c1t) in enumerate(per_scene_meta):
                    if not item_valid(tok, PARALLEL[phenom][s], D): continue
                    donor = cap_t(PARALLEL[phenom][s][D][0])
                    for j, l in enumerate(LAYERS):
                        P_opp[i, di, j] = margin(model, mods, tok, bos, pr_L, c0t, c1t, patch=(l, donor[l]))

            # 4) cross-same donors
            if same_D:
                for di, D in enumerate(same_D):
                    for i, (s, pr_L, c0t, c1t) in enumerate(per_scene_meta):
                        if not item_valid(tok, PARALLEL[phenom][s], D): continue
                        donor = cap_t(PARALLEL[phenom][s][D][0])
                        for j, l in enumerate(LAYERS):
                            P_same[i, di, j] = margin(model, mods, tok, bos, pr_L, c0t, c1t, patch=(l, donor[l]))

            kw = dict(scene_ids=np.array(scene_ids), y=np.zeros(n, int), m_clean=m_clean,
                      layers=np.array(LAYERS), HL=HL, HA_opp=HA_opp,
                      opp_D=np.array(opp_D), P_floor=P_floor, P_opp=P_opp)
            if P_same is not None:
                kw["same_D"] = np.array(same_D); kw["P_same"] = P_same
            else:
                kw["same_D"] = np.array([]); kw["P_same"] = np.array([])
            np.savez(f, **kw)
            acc = float((m_clean < 0).mean())  # canonical preferred means m_clean < 0
            print(f"saved {f.name} n={n} opp={opp_D} same={same_D} clean_acc={acc:.2f} {time.time()-t0:.0f}s")
    del model; gc.collect(); torch.cuda.empty_cache()

/home/anwoy/miniconda3/envs/ke/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 293/293 [00:00<00:00, 2478.61it/s]


saved bloom-1b7__adj__en.npz n=20 opp=['fr', 'es', 'ar'] same=['de', 'hi', 'zh'] clean_acc=1.00 12s
saved bloom-1b7__adj__fr.npz n=20 opp=['en', 'de', 'hi', 'zh'] same=['es', 'ar'] clean_acc=0.65 12s
saved bloom-1b7__adj__es.npz n=20 opp=['en', 'de', 'hi', 'zh'] same=['fr', 'ar'] clean_acc=0.70 12s
saved bloom-1b7__adj__de.npz n=20 opp=['fr', 'es', 'ar'] same=['en', 'hi', 'zh'] clean_acc=0.60 12s
saved bloom-1b7__adj__ar.npz n=20 opp=['en', 'de', 'hi', 'zh'] same=['fr', 'es'] clean_acc=1.00 12s
saved bloom-1b7__adj__hi.npz n=20 opp=['fr', 'es', 'ar'] same=['en', 'de', 'zh'] clean_acc=0.70 12s
saved bloom-1b7__adj__zh.npz n=20 opp=['fr', 'es', 'ar'] same=['en', 'de', 'hi'] clean_acc=0.85 12s
saved bloom-1b7__clause__en.npz n=15 opp=['hi'] same=['fr', 'es', 'de', 'ar', 'zh'] clean_acc=1.00 9s
saved bloom-1b7__clause__fr.npz n=15 opp=['hi'] same=['en', 'es', 'de', 'ar', 'zh'] clean_acc=1.00 9s
saved bloom-1b7__clause__es.npz n=15 opp=['hi'] same=['en', 'fr', 'de', 'ar', 'zh'] clean_acc=0.

Loading weights: 100%|██████████| 365/365 [00:00<00:00, 844.17it/s]


saved bloom-7b1__adj__en.npz n=20 opp=['fr', 'es', 'ar'] same=['de', 'hi', 'zh'] clean_acc=1.00 18s
saved bloom-7b1__adj__fr.npz n=20 opp=['en', 'de', 'hi', 'zh'] same=['es', 'ar'] clean_acc=0.65 19s
saved bloom-7b1__adj__es.npz n=20 opp=['en', 'de', 'hi', 'zh'] same=['fr', 'ar'] clean_acc=0.80 18s
saved bloom-7b1__adj__de.npz n=20 opp=['fr', 'es', 'ar'] same=['en', 'hi', 'zh'] clean_acc=0.75 18s
saved bloom-7b1__adj__ar.npz n=20 opp=['en', 'de', 'hi', 'zh'] same=['fr', 'es'] clean_acc=1.00 18s
saved bloom-7b1__adj__hi.npz n=20 opp=['fr', 'es', 'ar'] same=['en', 'de', 'zh'] clean_acc=0.60 18s
saved bloom-7b1__adj__zh.npz n=20 opp=['fr', 'es', 'ar'] same=['en', 'de', 'hi'] clean_acc=0.75 18s
saved bloom-7b1__clause__en.npz n=15 opp=['hi'] same=['fr', 'es', 'de', 'ar', 'zh'] clean_acc=1.00 14s
saved bloom-7b1__clause__fr.npz n=15 opp=['hi'] same=['en', 'es', 'de', 'ar', 'zh'] clean_acc=1.00 14s
saved bloom-7b1__clause__es.npz n=15 opp=['hi'] same=['en', 'fr', 'de', 'ar', 'zh'] clean_acc=

In [5]:
# CELL 5 — analysis: per-cell frac_flip (floor / cross-opp / cross-same), DTI, R_retr
import pandas as pd

rows = []
for f in sorted(OUT.glob("*.npz")):
    key, phenom, L = f.stem.split("__")
    z = np.load(f, allow_pickle=True)
    m_c, layers = z["m_clean"], z["layers"]
    HL, HA = z["HL"], z["HA_opp"]
    P_floor = z["P_floor"]
    P_opp_per_donor = z["P_opp"]       # [n, n_opp_D, n_layers]
    opp_D = list(z["opp_D"])
    has_same = z["P_same"].ndim == 3
    if has_same:
        P_same_per_donor = z["P_same"]
        same_D = list(z["same_D"])
    # average over donors per scene (ignore NaN from missing donor validity per scene)
    P_opp_scene  = np.nanmean(P_opp_per_donor, axis=1) if P_opp_per_donor.size else np.full_like(P_floor, np.nan)
    P_same_scene = np.nanmean(P_same_per_donor, axis=1) if has_same else np.full_like(P_floor, np.nan)

    # late-layer window: last 50% of LAYERS
    late_mask = layers >= layers.max() * 0.5
    def summ(P_scene):
        # median over late layers, per scene, then transfer_stats across scenes
        vals_per_scene = np.nanmedian(P_scene[:, late_mask], axis=1)
        ok = ~np.isnan(vals_per_scene) & ~np.isnan(m_c)
        if ok.sum() < 4: return None
        return transfer_stats(vals_per_scene[ok], m_c[ok])

    s_floor = summ(P_floor)
    s_opp   = summ(P_opp_scene)
    s_same  = summ(P_same_scene) if has_same else None

    # DTI
    dti = None
    if has_same:
        vals_opp  = np.nanmedian(P_opp_scene[:,  late_mask], axis=1)
        vals_same = np.nanmedian(P_same_scene[:, late_mask], axis=1)
        ok = ~np.isnan(vals_opp) & ~np.isnan(vals_same) & ~np.isnan(m_c)
        if ok.sum() >= 4:
            dti = dti_stats(vals_opp[ok], vals_same[ok], m_c[ok])

    # R_retr: anchor = cross-opp donor scenes. For adj, decision token is before any content → mark n/a.
    if phenom == "adj":
        r_med = np.nan
    else:
        rt = retrieval(HA, HL)
        late_rt = [r["R_retr"] for r in rt if r["layer"] >= layers.max() * 0.5]
        r_med = float(np.nanmedian(late_rt)) if any(np.isfinite(x) for x in late_rt) else np.nan

    clean_acc = float((m_c < 0).mean())
    row = dict(model=key, phenom=phenom, lang=L, n=int(len(m_c)), clean_acc=round(clean_acc, 2),
               opp_D=",".join(opp_D), same_D=",".join(same_D) if has_same else "—",
               floor=None if not s_floor else round(s_floor["frac_flip"], 2),
               floor_lo=None if not s_floor else round(s_floor["lo"], 2),
               floor_hi=None if not s_floor else round(s_floor["hi"], 2),
               opp=None if not s_opp else round(s_opp["frac_flip"], 2),
               opp_lo=None if not s_opp else round(s_opp["lo"], 2),
               opp_hi=None if not s_opp else round(s_opp["hi"], 2),
               same=None if not s_same else round(s_same["frac_flip"], 2),
               same_lo=None if not s_same else round(s_same["lo"], 2),
               same_hi=None if not s_same else round(s_same["hi"], 2),
               DTI=None if not dti else round(dti["dti"], 2),
               DTI_lo=None if not dti else round(dti["lo"], 2),
               DTI_hi=None if not dti else round(dti["hi"], 2),
               R_retr=round(r_med, 2) if np.isfinite(r_med) else "n/a")
    rows.append(row)

df = pd.DataFrame(rows).sort_values(["model","phenom","lang"]).reset_index(drop=True)
df.to_csv(OUT / "exp26_summary.csv", index=False)

def fmt(v, lo, hi):
    if v is None or pd.isna(v): return "        —        "
    return f"{v:+.2f}[{lo:+.2f},{hi:+.2f}]"

print("\n=== Calibrated word-order patching per cell ===")
print("floor = same-L different-scene (sanity, should be ~0)")
print("opp   = cross-opposite-order donor (what Exp22/25 reported)")
print("same  = cross-same-order donor (controls for foreign-state noise)")
print("DTI   = opp - same (order-specific transfer; CI excluding 0 = real signal)")
print("* on DTI = CI excludes 0\n")

for _, r in df.iterrows():
    star = " *" if (pd.notna(r["DTI_lo"]) and pd.notna(r["DTI_hi"]) and (r["DTI_lo"] > 0 or r["DTI_hi"] < 0)) else "  "
    print(f"{r['model']:9s} {r['phenom']:6s} {r['lang']}  n={r['n']:2d}  acc={r['clean_acc']:.2f}  "
          f"floor={fmt(r['floor'], r['floor_lo'], r['floor_hi'])}  "
          f"opp={fmt(r['opp'], r['opp_lo'], r['opp_hi'])}  "
          f"same={fmt(r['same'], r['same_lo'], r['same_hi'])}  "
          f"DTI={fmt(r['DTI'], r['DTI_lo'], r['DTI_hi'])}{star}  "
          f"R_retr={r['R_retr']}")

# verdict table filtered to cells with CI-excludes-0 DTI
sig = df[df.apply(lambda r: pd.notna(r["DTI_lo"]) and pd.notna(r["DTI_hi"]) and (r["DTI_lo"] > 0 or r["DTI_hi"] < 0), axis=1)]
print(f"\n=== Cells with significant DTI (CI excludes 0): {len(sig)} / {len(df)} ===")
if len(sig):
    print(sig[["model","phenom","lang","DTI","DTI_lo","DTI_hi","R_retr"]].to_string(index=False))
else:
    print("None. Either word order does not transfer cross-lingually in BLOOM, or n is too low.")

/tmp/ipykernel_2386739/2802152353.py:19: RuntimeWarning: Mean of empty slice
  P_same_scene = np.nanmean(P_same_per_donor, axis=1) if has_same else np.full_like(P_floor, np.nan)
/tmp/ipykernel_2386739/2802152353.py:25: RuntimeWarning: All-NaN slice encountered
  vals_per_scene = np.nanmedian(P_scene[:, late_mask], axis=1)
/tmp/ipykernel_2386739/2802152353.py:38: RuntimeWarning: All-NaN slice encountered
  vals_same = np.nanmedian(P_same_scene[:, late_mask], axis=1)



=== Calibrated word-order patching per cell ===
floor = same-L different-scene (sanity, should be ~0)
opp   = cross-opposite-order donor (what Exp22/25 reported)
same  = cross-same-order donor (controls for foreign-state noise)
DTI   = opp - same (order-specific transfer; CI excluding 0 = real signal)
* on DTI = CI excludes 0

bloom-1b7 adj    ar  n=20  acc=1.00  floor=+0.01[-0.02,+0.04]  opp=-0.00[-0.10,+0.10]  same=-0.03[-0.15,+0.08]  DTI=+0.03[-0.02,+0.09]    R_retr=n/a
bloom-1b7 adj    de  n=20  acc=0.60  floor=-0.07[-0.19,+0.03]  opp=-0.26[-0.64,+0.02]  same=+0.01[-0.23,+0.27]  DTI=-0.27[-0.52,-0.13] *  R_retr=n/a
bloom-1b7 adj    en  n=20  acc=1.00  floor=+0.02[-0.05,+0.09]  opp=+0.08[-0.02,+0.15]  same=+0.06[-0.01,+0.13]  DTI=+0.02[-0.06,+0.08]    R_retr=n/a
bloom-1b7 adj    es  n=20  acc=0.70  floor=+0.11[-0.02,+0.26]  opp=+0.21[+0.10,+0.32]  same=+0.01[-0.09,+0.11]  DTI=+0.20[+0.13,+0.29] *  R_retr=n/a
bloom-1b7 adj    fr  n=20  acc=0.65  floor=+0.08[+0.00,+0.16]  opp=+0.13[+